# DRLB Value Non-Zero Check

This notebook runs the DRLB pipeline on the subsample config and checks that the Optuna objective value is no longer `0.0`.

It uses the full subsample for training and the deterministic 10% eval slice configured in `SubsampleRND42N10Config`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

notebook_dir = Path().resolve()
example_notebooks_dir = notebook_dir.parent.parent
bat_autobidding_dir = example_notebooks_dir.parent

if str(example_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(example_notebooks_dir))
if str(bat_autobidding_dir) not in sys.path:
    sys.path.insert(0, str(bat_autobidding_dir))

from experiments.exp_configs import SubsampleRND42N10Config
from experiments.drlb_experiment import opt_search_drlb, train_best_drlb, evaluate_drlb


In [ ]:
config = SubsampleRND42N10Config()
config.ensure_artifact_dirs()

stats_df = pd.read_csv(config.data_config["train"]["stats_path"])
n_trials = 1

print("Train stats rows:", len(stats_df))
print("Eval fraction:", config.eval_campaign_fraction)
print("Eval slice seed:", config.eval_slice_seed)

In [ ]:
study, best_params_path = opt_search_drlb(
    config=config,
    stats_df=stats_df,
    n_trials=n_trials,
    verbose=False,
)

best_value = float(study.best_trial.value)
print("Best trial value:", best_value)
assert best_value != 0.0, "DRLB best trial value is still 0.0"
best_params_path

In [ ]:
best_model_path = config.best_models_dir / f"drlb_{config.metric.lower()}_{config.auction_mode}.pt"

_ = train_best_drlb(
    stats_df=stats_df,
    best_params_path=str(best_params_path),
    model_path=str(best_model_path),
    config=config,
    verbose=False,
)

res = evaluate_drlb(
    config=config,
    model_path=str(best_model_path),
    best_params_path=str(best_params_path),
    verbose=False,
)

res["score"]

In [ ]:
import json

metrics_path = config.outputs_dir / "drlb_eval_metrics.json"
diagnostics_path = config.outputs_dir / "drlb_train_training_diagnostics.csv"

with open(metrics_path) as f:
    metrics = json.load(f)

print(metrics)
pd.read_csv(diagnostics_path).tail(10)